# 第79课 · 让相似的歌在向量空间相遇——对比学习（contrastive learning）与音乐嵌入（embedding）

**目标**：训练对比学习音乐编码器——同歌近、异歌远，输出固定长 embedding。

> 回链 L10 相似度；说清向量来自本课训练还是预训练权重。

🔗 **Aurora 连接**：本节的三个实现已落入 `aurora.music.embed`，可通过 `from aurora.music import MusicEncoder, triplet_loss, nt_xent_loss` 直接使用（需安装 `pip install aurora[music]`）。检索侧的 `aurora.music.knn_search` 将在 L80 中使用这些向量完成最近邻查询。

← **上一课**　[L78 · 节拍追踪从零实现](L78_beat_tracking.ipynb)

> 上节课学习了 **节拍追踪从零实现**：onset 包络、自相关与 BPM 估计。  
> 本课将探讨 **音乐嵌入向量（embedding）**。

## 本课剧情：Shazam 的弟弟——让 AI 判断两首歌是否同风格

Shazam 用指纹匹配识别歌曲。但如果你问"帮我找10首类似《Bohemian Rhapsody》风格的歌"，Shazam 做不到——它不理解"风格相似"。

**对比学习（Contrastive Learning）**的思路：把音乐片段映射到一个向量空间，让"相似的音乐靠近，不同的音乐拉远"。

```
《Bohemian Rhapsody》片段1 ─────┐
《Bohemian Rhapsody》片段2 ──── → 向量空间中靠近  ← 同一首歌的不同片段
《Numb》（不同风格）──────────── → 向量空间中拉远
```

**Triplet Loss**：每次用三元组训练：
- **Anchor**（锚点）：参考样本
- **Positive**（正样本）：与anchor同类/同风格
- **Negative**（负样本）：与anchor不同类

```
L = max(d(a,p) - d(a,n) + margin, 0)
```
- 若 `d(a,p) < d(a,n) - margin`（正样本已经比负样本近 margin）→ loss=0（训练完成）
- 否则 → 梯度推动 anchor/positive 靠近、anchor/negative 远离

**margin 的作用**：防止模型"作弊"——把所有向量都映射到同一点，loss=0 但毫无区分力。

**本节流程**：实现 triplet_loss → MusicEncoder（CNN+AdaptiveAvgPool）→ L2 归一化 → 聚类可视化。

In [1]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [2]:
import torch
import torch.nn as nn
import numpy as np

### 🤔 先想一想：为什么"拉近/推远"就能学会"风格"？

先别看公式。想象编码器只被允许听两条命令：

- **同一首歌的两个片段** → 在向量空间里靠近一点；
- **不同歌的片段** → 离远一点。

反复执行成千上万次后，编码器要**同时**满足所有这些远近约束，唯一的出路是：自己去发现"哪些声学特征能区分歌曲与风格"，并把它们编码进向量的方向里。没人告诉它"什么是摇滚"——它是被这些远近关系"逼"出来的。

这就是对比学习的核心直觉：**不定义类别，只定义"谁和谁像"，让几何结构自己长出语义。** 下面的 Triplet Loss，就是把这两条命令写成一个可求导的公式。

## 1. 对比学习：Triplet Loss

每次训练以三元组 (anchor, positive, negative) 为单位：
- `anchor`：参考片段
- `positive`：与 anchor 来自同一首歌的另一片段
- `negative`：来自不同歌的片段

**Triplet Loss 公式**：

```
L = mean( max(0,  d(anchor, pos) - d(anchor, neg) + margin) )
```

其中 `d` 是 L2 距离，`margin` 是一个安全边距（典型值 0.2）。当 `d_neg - d_pos > margin` 时 loss 为 0，说明负样本（negative sample）已足够远，不再产生梯度。

In [3]:
# 直觉演示：loss 面
import torch

margin = 0.2
d_pos_vals = torch.linspace(0, 2, 50)
d_neg_fixed = 1.0  # 固定负样本距离

loss_vals = torch.clamp(d_pos_vals - d_neg_fixed + margin, min=0)

print("d_pos\t\td_neg\t\tloss")
for i in [0, 12, 25, 37, 49]:
    print(f"{d_pos_vals[i]:.2f}\t\t{d_neg_fixed:.2f}\t\t{loss_vals[i]:.4f}")

d_pos		d_neg		loss
0.00		1.00		0.0000
0.49		1.00		0.0000
1.02		1.00		0.2204
1.51		1.00		0.7102
2.00		1.00		1.2000


## 2. MusicEncoder：Mel → 固定长度向量

输入：mel 特征矩阵，形状 `(B, 1, T, n_mels)`，时间轴可变长。
架构：
```
Conv2d(1→16, k=3) → ReLU → MaxPool(2,2)
Conv2d(16→32, k=3) → ReLU → AdaptiveAvgPool → 全局压平
Linear(32*n_mels//4, embed_dim=128)
```
`AdaptiveAvgPool2d((1, n_mels//4))` 把时间轴压成 1，使输出长度与输入时长无关——这是处理可变长音频的关键。

In [4]:
class MusicEncoder(nn.Module):
    def __init__(self, n_mels=64, embed_dim=128):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                       # T//2, n_mels//2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, n_mels // 4)),   # 1, n_mels//4
        )
        self.fc = nn.Linear(32 * (n_mels // 4), embed_dim)

    def forward(self, x):                             # x: (B,1,T,n_mels)
        h = self.cnn(x)                               # (B,32,1,n_mels//4)
        h = h.flatten(1)                              # (B, 32*n_mels//4)
        return self.fc(h)                             # (B, embed_dim)

# 验证输入输出形状
enc = MusicEncoder(n_mels=64, embed_dim=128)
dummy = torch.randn(4, 1, 100, 64)   # batch=4, T=100帧, n_mels=64
out = enc(dummy)
print(f"输入形状: {dummy.shape}  →  输出形状: {out.shape}")
assert out.shape == (4, 128), "形状不对"
print("✅ MusicEncoder 形状验证通过")

输入形状: torch.Size([4, 1, 100, 64])  →  输出形状: torch.Size([4, 128])
✅ MusicEncoder 形状验证通过


## 3. L2 归一化：让向量落在单位球面（unit hypersphere）

训练后对输出做 L2 归一化：

```
v_norm = v / ||v||_2
```

归一化后向量模长 = 1，此时：
- **余弦相似度（cosine similarity）** = `v_a · v_b`（直接点积，无需除模长）
- L2 距离与余弦距离单调等价：`||a-b||^2 = 2 - 2*(a·b)`

这让 ANN 检索（FAISS 内积索引）可直接替代余弦搜索，速度更快。

In [5]:
def l2_normalize(x: torch.Tensor) -> torch.Tensor:
    """对最后一维做 L2 归一化。"""
    return x / (x.norm(dim=-1, keepdim=True) + 1e-8)

v = torch.randn(3, 128)
v_n = l2_normalize(v)
norms = v_n.norm(dim=-1)
print(f"归一化后各向量的模长: {norms.tolist()}")
assert torch.allclose(norms, torch.ones(3), atol=1e-5)
print("✅ L2 归一化验证通过")

归一化后各向量的模长: [0.9999999403953552, 1.0, 1.0]
✅ L2 归一化验证通过


## 4. ✏️ 实现 `triplet_loss(anchor, positive, negative, margin=0.2)`

**签名**：
```python
def triplet_loss(anchor: torch.Tensor, positive: torch.Tensor,
                 negative: torch.Tensor, margin: float = 0.2) -> torch.Tensor:
    # anchor/positive/negative: (B, embed_dim) L2-normalized embeddings
    # 返回: scalar Tensor，batch平均loss
```

**3步实现**：

| 步骤 | 操作 | PyTorch 工具 |
|---|---|---|
| 1 | 计算欧氏距离 | `d_pos = (anchor - positive).norm(dim=1)` |
| 2 | `loss_per_sample = max(d_pos - d_neg + margin, 0)` | `torch.clamp(..., min=0)` |
| 3 | 返回 batch 平均 | `.mean()` |

**验收标准**：
- `anchor==positive, d(a,n)=2.0, margin=0.2` → loss ≈ 0（正样本已比负样本近 margin）
- `d(a,p)=1.5, d(a,n)=1.0, margin=0.5` → loss = max(1.5-1.0+0.5, 0) = 1.0
- 输出是 scalar Tensor（shape=()）

In [6]:
def triplet_loss(anchor, positive, negative, margin=0.2):
    """
    Triplet loss for contrastive learning.
    Args:
        anchor, positive, negative: torch.Tensor, shape (B, d)
        margin: float, safety margin
    Returns:
        scalar Tensor
    """
    # ✏️ TODO: 依次实现 d_pos、d_neg、loss 三步（完成后删除 raise 行）
    raise NotImplementedError("TODO: 计算 d_pos, d_neg, loss")
    # d_pos = torch.norm(anchor - positive, dim=-1)

    # ✏️ TODO: 计算 d_neg = L2(anchor, negative) 每行距离
    # d_neg = torch.norm(anchor - negative, dim=-1)

    # ✏️ TODO: loss = clamp(d_pos - d_neg + margin, min=0).mean()
    # loss = torch.clamp(d_pos - d_neg + margin, min=0.0).mean()

    return loss

In [7]:
# 检查1：anchor==positive, negative 很远 → loss ≈ 0
a = torch.zeros(4, 8)
p = torch.zeros(4, 8)
n = torch.ones(4, 8) * 10
try:
    _loss1 = triplet_loss(a, p, n)
except (NotImplementedError, TypeError):
    _loss1 = None
if _loss1 is None:
    print('⬜ 请先实现 triplet_loss 的三个 TODO 项')
else:
    val = _loss1.item()
    assert val < 0.01, f"期望 loss < 0.01，实际 {val:.4f}"
    print(f"✅ 检查1通过：loss = {val:.6f}（应接近0）")

    # 检查2：anchor==negative, positive 很远 → loss ≈ d_pos + margin
    a2 = torch.zeros(4, 8)
    p2 = torch.ones(4, 8) * 10
    n2 = torch.zeros(4, 8)
    _loss2 = triplet_loss(a2, p2, n2)
    if _loss2 is None:
        print('⬜ 请先实现 triplet_loss')
    else:
        val2 = _loss2.item()
        d_pos_expected = (8 ** 0.5) * 10   # ||ones*10|| for dim=8
        expected = d_pos_expected + 0.2
        assert abs(val2 - expected) < 0.01, f"期望 {expected:.4f}，实际 {val2:.4f}"
        print(f"✅ 检查2通过：loss = {val2:.4f}（应 ≈ {expected:.4f}）")


⬜ 请先实现 triplet_loss 的三个 TODO 项


## 5. 参数实验：模拟三类音乐的 Embedding 聚类

**设置**：用随机生成的 mel 特征模拟 3 种风格（摇滚 / 爵士 / 古典），各 20 首歌，训练 50 轮。

**关键参数**：
- `margin=0.2`：过小 → 负样本靠得太近；过大 → 梯度消失（hard negative 不够）
- `embed_dim=128`：降到 16 可看到更明显聚类（维数诅咒（curse of dimensionality）减弱）
- `lr=1e-3`：收敛曲线若抖动，调低至 1e-4

**预期现象**：训练结束后 t-SNE 图中三类点应形成三簇，簇间距明显大于簇内距。

## 🎯 未来的回报 (Future Payoff)

今天你亲手搞懂的 **嵌入（把对象映射成向量、用几何距离表示语义相似）**，会在 **L80 / L83 / L88** 再次出现——L80 直接拿这些向量做最近邻检索；L83 的 Transformer 把每个 token 也变成 embedding；L88 的 TF-IDF 检索则是它"无神经网络"的表亲。**"用向量的方向表示含义"是贯穿整个现代 AI 的同一个思想**，音乐只是它的第一个练手场。

In [8]:
import torch
import torch.nn as nn
import numpy as np

# ---------- 模拟数据 ----------
torch.manual_seed(42)
np.random.seed(42)

n_mels, T, embed_dim = 64, 50, 128
n_classes, n_per_class = 3, 20
# 每类中心不同，以使 CNN 能学到差异
class_centers = [
    torch.randn(1, 1, T, n_mels) * 2 + i * 3
    for i in range(n_classes)
]

def make_mel(class_id):
    """生成属于 class_id 的 mel 特征（加小噪声）。"""
    return class_centers[class_id] + torch.randn(1, 1, T, n_mels) * 0.3

all_mels = []  # (class_id, mel_tensor)
for c in range(n_classes):
    for _ in range(n_per_class):
        all_mels.append((c, make_mel(c)))

# ---------- 模型 & 优化器 ----------
encoder = MusicEncoder(n_mels=n_mels, embed_dim=embed_dim)
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-3)

# ---------- 训练前 stub 检查 ----------
_probe_a = l2_normalize(encoder(make_mel(0)))
_probe_p = l2_normalize(encoder(make_mel(0)))
_probe_n = l2_normalize(encoder(make_mel(1)))
try:
    _probe_loss = triplet_loss(_probe_a, _probe_p, _probe_n)
except (NotImplementedError, TypeError):
    _probe_loss = None

if _probe_loss is None:
    print('⬜ 请先实现 triplet_loss，再运行训练循环')
else:
    # ---------- 训练 50 轮 ----------
    losses = []
    for epoch in range(50):
        epoch_loss = 0.0
        indices = torch.randperm(len(all_mels))
        for idx in indices:
            c_a, mel_a = all_mels[idx]
            # positive：同类随机另一首，但不能抽到自己
            pos_idx = torch.randint(n_per_class, (1,)).item()
            while c_a * n_per_class + pos_idx == idx.item():
                pos_idx = torch.randint(n_per_class, (1,)).item()
            _, mel_p = all_mels[c_a * n_per_class + pos_idx]
            # negative：不同类随机一首
            neg_class = (c_a + torch.randint(1, n_classes, (1,)).item()) % n_classes
            neg_idx = torch.randint(n_per_class, (1,)).item()
            _, mel_n = all_mels[neg_class * n_per_class + neg_idx]

            a = l2_normalize(encoder(mel_a))
            p = l2_normalize(encoder(mel_p))
            n_ = l2_normalize(encoder(mel_n))

            loss = triplet_loss(a, p, n_, margin=0.2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        losses.append(epoch_loss / len(all_mels))
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}/50  loss={losses[-1]:.4f}")

    # ---------- t-SNE 可视化 ----------
    encoder.eval()
    embeds, labels = [], []
    with torch.no_grad():
        for c, mel in all_mels:
            e = l2_normalize(encoder(mel)).squeeze(0).numpy()
            embeds.append(e)
            labels.append(c)
    embeds = np.array(embeds)

    try:
        from sklearn.manifold import TSNE
        import matplotlib.pyplot as plt

        tsne = TSNE(n_components=2, perplexity=10, random_state=42)
        proj = tsne.fit_transform(embeds)

        colors = ['#e74c3c', '#3498db', '#2ecc71']
        names  = ['摇滚', '爵士', '古典']
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # t-SNE 聚类图
        for c in range(n_classes):
            mask = np.array(labels) == c
            axes[0].scatter(proj[mask, 0], proj[mask, 1],
                            c=colors[c], label=names[c], s=60, alpha=0.8)
        axes[0].set_title("t-SNE：三类音乐 Embedding 聚类")
        axes[0].legend()

        # 训练 loss 曲线
        axes[1].plot(losses, color='#8e44ad')
        axes[1].set_title("训练 Loss 曲线（50 轮）")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Triplet Loss")

        plt.tight_layout()
        plt.show()
        print("✅ 可视化完成")
    except ImportError:
        print("sklearn / matplotlib 未安装，跳过可视化")
        print(f"最终 loss: {losses[-1]:.4f}")

⬜ 请先实现 triplet_loss，再运行训练循环


## 本课收束

本节实现了 `triplet_loss`，用 L2 距离把同一首歌的向量拉近、把不同歌的向量推远，并通过 `MusicEncoder`（CNN + AdaptiveAvgPool）把可变长 mel 特征压缩为固定 128 维向量。输出经 `l2_normalize` 落在单位球面，让余弦相似度直接变成点积，帮后面的 FAISS 检索省下计算。第六节引入了 **NT-Xent Loss**（SimCLR 变体）：把整个 batch 中其余所有样本都当负样本，信息密度远超 Triplet Loss，是工业级对比学习的主流选择。第七节展示了由此通向 **CLAP**（跨模态文字-音频对比学习）和 MusicGen/Suno 的技术路径——本节编码器是其音频侧的雏形。这套编码器构成 `aurora.music` 的语义骨干——下一节 **L80** 将基于此向量构建最近邻检索索引，实现毫秒级“找相似歌曲”。

## 6. NT-Xent Loss（SimCLR）：批量内对比，比 Triplet 更强

Triplet Loss 每步只看一个负样本；**NT-Xent**（Normalized Temperature-Scaled Cross-Entropy）把整个 batch 中其余所有样本都当负样本，信息密度指数级提升。

设 batch 大小 B，每个 `z1[i]` 对应正对 `z2[i]`（同一首歌的两个增强片段）：

```
L_i = -log [ exp(sim(z1_i, z2_i) / τ) / Σ_{k≠i} exp(sim(z1_i, z_k) / τ) ]
```

`τ`（temperature）是关键超参（典型值 0.07–0.2）：越小越惩罚 hard negatives，梯度越集中。这是 **SimCLR** 的核心损失，也是 **CLAP**（对比语言-音频预训练）的数学基础。

**Hard Negative Mining**：NT-Xent 天然包含 hard negative 效应——分母 Σ 里相似度越高的负样本，梯度权重越大。显式挖掘（MoCo v3 风格）通过更大 batch 或 memory bank 扩充负样本库，对 aurora.music 的实际影响：batch size 64 → 128 可明显提升 embedding 分辨力。

In [9]:
import torch
import torch.nn.functional as F

def nt_xent_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    """NT-Xent / SimCLR 对比损失。

    z1, z2 : shape (B, d)，每对 (z1[i], z2[i]) 是同一首歌的两个增强片段（正对）。
    两者应已 L2 归一化；其余 2B-2 个样本为负对。
    """
    B = z1.shape[0]
    z = torch.cat([z1, z2], dim=0)                 # (2B, d)
    sim = (z @ z.T) / temperature                   # (2B, 2B)
    sim.fill_diagonal_(float('-inf'))               # 屏蔽自身（不与自己对比）
    # z1[i] 的正对是 z2[i]（索引 B+i）；z2[i] 的正对是 z1[i]（索引 i）
    labels = torch.cat([torch.arange(B, 2 * B),
                        torch.arange(0, B)]).to(z.device)
    return F.cross_entropy(sim, labels)


# ── 验证 1：正对几乎相同 → loss 应趋近理论下界 ─────────────────────────────────
B, d = 8, 64
torch.manual_seed(0)
z1 = F.normalize(torch.randn(B, d), dim=1)
z2 = F.normalize(z1 + torch.randn(B, d) * 0.01, dim=1)   # 轻微扰动
loss_good = nt_xent_loss(z1, z2, temperature=0.07)

# ── 验证 2：完全随机（负对难以区分）→ loss 应远高于正对相同 ──────────────────
# 理论上界 log(2B-1) 仅在 τ=1 成立（此时相似度≈0，退化为均匀分类）；
# 低温 τ=0.07 会放大 logits，使随机 loss 明显高于该上界——但概念一致：随机→高 loss。
z2_rand = F.normalize(torch.randn(B, d), dim=1)
loss_rand      = nt_xent_loss(z1, z2_rand, temperature=0.07)
loss_rand_tau1 = nt_xent_loss(z1, z2_rand, temperature=1.0)
upper = torch.log(torch.tensor(2.0 * B - 1))

print(f"正对≈相同 loss        = {loss_good.item():.4f}  (预期低)")
print(f"完全随机 loss(τ=0.07) = {loss_rand.item():.4f}  (低温放大，高于上界)")
print(f"完全随机 loss(τ=1.0)  = {loss_rand_tau1.item():.4f}  (预期 ≈ log({2*B-1})={upper:.2f})")
assert loss_good < loss_rand, "正对相似时 loss 应低于随机"

# ── 温度敏感性实验 ────────────────────────────────────────────────────────────
print("\n温度对 loss 的影响（正对相同，batch=16）:")
B2, d2 = 16, 64
z1b = F.normalize(torch.randn(B2, d2), dim=1)
z2b = F.normalize(z1b + torch.randn_like(z1b) * 0.05, dim=1)
for tau in [0.05, 0.1, 0.2, 0.5]:
    l = nt_xent_loss(z1b, z2b, temperature=tau).item()
    print(f"  τ={tau:.2f}  loss={l:.4f}")

print("\n✅ NT-Xent 验证通过")

正对≈相同 loss        = 0.0000  (预期低)
完全随机 loss(τ=0.07) = 5.6353  (低温放大，高于上界)
完全随机 loss(τ=1.0)  = 2.7963  (预期 ≈ log(15)=2.71)

温度对 loss 的影响（正对相同，batch=16）:
  τ=0.05  loss=0.0000
  τ=0.10  loss=0.0067
  τ=0.20  loss=0.3197
  τ=0.50  loss=1.7880

✅ NT-Xent 验证通过


## 7. CLAP 与通向 MusicGen/Suno 的路径

### CLAP：跨模态 NT-Xent

**CLAP**（Contrastive Language-Audio Pre-training）把 NT-Xent 扩展到**文字-音频**跨模态：

```
loss = NT-Xent(audio_encoder(音频), text_encoder(文字标签), τ=0.07)
```

训练完成后，"upbeat jazz piano"的文字 embedding 与真正的爵士钢琴音频 embedding 距离最近。
LAION 的 `larger_clap_general` 在 HuggingFace 上直接可用（约 900MB）。

### Suno/MusicGen 的核心三层

```
[文字描述]
    ↓  T5 / CLAP text encoder → text embedding
[Transformer Decoder]  ← cross-attention 条件化
    ↓  自回归预测 token
[EnCodec token 序列]  (50 帧/s × 4 码书)
    ↓  EnCodec decoder
[32kHz 音频波形]
```

aurora.music 的 `MusicEncoder + NT-Xent` 是第一层的**音频侧**雏形；
理解了这套逻辑，就理解了 MusicGen 整体架构——区别只在规模（数百万小时训练数据）和文字接入（T5/CLAP vs 本节的无文字条件）。

→ **下一步**：打开 L80 · 向量相似度检索，用余弦相似度在 embedding 空间里找出最相似的 top-k 曲目，验证三元组训练的效果。

## ✏️ 白板挑战：对比学习手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：Triplet Loss 中 margin 的作用是什么？若 margin=0 会发生什么？  
（margin要求positive比negative至少近margin距离；margin=0时模型可把所有向量映射到同一点使loss=0，失去区分能力）

**问 2**：若 d(a,p)=0.3，d(a,n)=0.8，margin=0.2，Triplet Loss=？  
（max(0.3-0.8+0.2,0)=max(-0.3,0)=0.0，正样本已经足够近，梯度为0）

**问 3**：若 d(a,p)=0.9，d(a,n)=0.5，margin=0.5，Triplet Loss=？  
（max(0.9-0.5+0.5,0)=max(0.9,0)=0.9，负样本太近了，需要推远）

**问 4**：L2 归一化（v / ||v||）后，所有向量落在哪个几何结构上？欧氏距离和余弦相似度有什么关系？  
（单位超球面；L2归一化后 ||a-b||²=2(1-cos_sim)，欧氏距离和余弦相似度单调对应，可互换使用）

**问 5**：NT-Xent Loss 相比 Triplet Loss 的主要优势是什么？  
（每次利用整个 batch 的负样本（B-1 个），而 Triplet 每次只看 1 个负样本；batch 越大，负样本越难，对比效果越强）

推导完成后运行下方格验证。

In [10]:
# ✏️ 对答案格
import torch
import numpy as np

# 问1：margin=0退化演示
a = torch.zeros(4, 8)   # anchor
p = torch.zeros(4, 8)   # positive (相同点)
n = torch.zeros(4, 8)   # negative (也相同点！)
try:
    loss_zero_margin = triplet_loss(a, p, n, margin=0.0)
    assert abs(loss_zero_margin.item()) < 1e-6, "全相同时loss应=0"
    print(f"Q1 ✅  margin=0, 所有向量相同 → loss={loss_zero_margin.item():.4f}（模型可作弊：全映射同一点）")
except (NotImplementedError, TypeError):
    print(f"Q1 ✅  margin防作弊：无margin时全映射同一点loss=0，gradient消失（待实现后验证）")

# 问2：d(a,p)=0.3, d(a,n)=0.8, margin=0.2 → loss=0
try:
    embed_dim = 8
    # 构造满足指定距离的向量
    a2 = torch.zeros(1, embed_dim)
    p2 = torch.zeros(1, embed_dim); p2[0, 0] = 0.3  # d(a,p)=0.3
    n2 = torch.zeros(1, embed_dim); n2[0, 0] = 0.8  # d(a,n)=0.8
    loss_q2 = triplet_loss(a2, p2, n2, margin=0.2)
    expected_q2 = max(0.3 - 0.8 + 0.2, 0)
    assert abs(loss_q2.item() - expected_q2) < 1e-4, f"loss应={expected_q2:.1f}，得{loss_q2.item():.4f}"
    print(f"Q2 ✅  d(a,p)=0.3,d(a,n)=0.8,margin=0.2: loss=max(0.3-0.8+0.2,0)={expected_q2:.1f}（正样本已够近）")
except (NotImplementedError, TypeError):
    print(f"Q2 ✅  max(0.3-0.8+0.2,0)=max(-0.3,0)=0.0（正样本已比负样本近margin，无梯度）（待实现）")

# 问3：d(a,p)=0.9, d(a,n)=0.5, margin=0.5 → loss=0.9
try:
    a3 = torch.zeros(1, embed_dim)
    p3 = torch.zeros(1, embed_dim); p3[0, 0] = 0.9
    n3 = torch.zeros(1, embed_dim); n3[0, 0] = 0.5
    loss_q3 = triplet_loss(a3, p3, n3, margin=0.5)
    expected_q3 = max(0.9 - 0.5 + 0.5, 0)
    assert abs(loss_q3.item() - expected_q3) < 1e-4, f"loss应={expected_q3:.1f}，得{loss_q3.item():.4f}"
    print(f"Q3 ✅  d(a,p)=0.9,d(a,n)=0.5,margin=0.5: loss=max(0.9-0.5+0.5,0)={expected_q3:.1f}（负样本太近！）")
except (NotImplementedError, TypeError):
    print(f"Q3 ✅  max(0.9-0.5+0.5,0)=0.9（负样本比正样本近，需要推远）（待实现）")

# 问4：L2归一化与余弦相似度关系
v1 = torch.tensor([[3.0, 4.0]])  # ||v1||=5
v2 = torch.tensor([[0.0, 1.0]])  # ||v2||=1
v1_norm = v1 / v1.norm()
v2_norm = v2 / v2.norm()
cos_sim = (v1_norm * v2_norm).sum().item()
euclidean_sq = ((v1_norm - v2_norm) ** 2).sum().item()
expected_sq = 2 * (1 - cos_sim)
assert abs(euclidean_sq - expected_sq) < 1e-5, f"||a-b||²=2(1-cos)应成立"
print(f"Q4 ✅  L2归一化后: ||a-b||²={euclidean_sq:.4f}, 2(1-cos)={expected_sq:.4f}（完全一致，欧氏↔余弦可互换）")

# 问5：NT-Xent 负样本数量
batch_size = 32
n_negatives_triplet = 1   # Triplet: 每次1个负样本
n_negatives_ntxent = batch_size - 1  # NT-Xent: B-1个负样本
print(f"Q5 ✅  batch={batch_size}: Triplet={n_negatives_triplet}个负样本 vs NT-Xent={n_negatives_ntxent}个（更多→更强对比信号）")
print("\n🎉 对比学习白板挑战通过！")

Q1 ✅  margin防作弊：无margin时全映射同一点loss=0，gradient消失（待实现后验证）
Q2 ✅  max(0.3-0.8+0.2,0)=max(-0.3,0)=0.0（正样本已比负样本近margin，无梯度）（待实现）
Q3 ✅  max(0.9-0.5+0.5,0)=0.9（负样本比正样本近，需要推远）（待实现）
Q4 ✅  L2归一化后: ||a-b||²=0.4000, 2(1-cos)=0.4000（完全一致，欧氏↔余弦可互换）
Q5 ✅  batch=32: Triplet=1个负样本 vs NT-Xent=31个（更多→更强对比信号）

🎉 对比学习白板挑战通过！


In [ ]:
# ✏️ 本课自评
l79_review = {
    "contrastive_learning_idea":  None,  # 理解对比学习：相似靠近、不同拉远，用triplet三元组？True/False
    "margin_role_understood":     None,  # 理解margin防作弊：没有margin时模型可把所有向量映射同一点？True/False
    "triplet_loss_impl":          None,  # triplet_loss()实现正确，3个验收标准通过？True/False
    "l2_norm_geometry":           None,  # 理解L2归一化→单位球面，||a-b||²=2(1-cos_sim)？True/False
    "whiteboard_passed":          None,  # 白板挑战5道手推全部完成？True/False
}

unfilled = [k for k, v in l79_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l79_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L79 全部通关！进入 L80：向量相似度检索')

---

→ **下一课**　[L80 · 向量相似度检索](L80_similarity.ipynb)

> 下节课将学习 **向量相似度检索**：余弦相似度 vs 点积 vs L2，纯 NumPy k-NN 实现。